# rackmains / ups / battery

**Intent**
- Outputs `guard_{shed_max,halt_max,mains_stable,halt_delay}_seconds`, the shutdown guard's required thresholds.
- Gives them as constants from its first release, through the project path every later model uses.
- Carries optional outputs the fitted model will fill.
- Those are `guard_{shed_lead,halt_lead}_seconds` and `battery_remaining_{pct,seconds}`.
- The rest are `model_{energy_used_wh,history_known,recalibrate,state_out}`, with the state carried by the caller.

**Data**
- `golden` is the fixed samples and expected outputs.
- `training`, `validation` and `backtest` follow with the fit, from the outage CSV files and the plug pairs.
- The first drain to the knee, trial 5, is the data the fit needs.

**Status**
- The default constants project is built and is the first release.
- No fitted model is built, and none ships until a drain to the knee exists.

**Future**
- Overall: the work still to do before the first release:
  - Find the time to the knee from trial 5, and fit `usable_wh` from the load through the drain, shaded to the safe side.
  - Build the plug pairs into a load to watts map, and set the thresholds from the knee and the measured host times.
  - Write golden rows from at least three sequences worked by hand from the parameters.
  - Backtest every outage file row by row, so the estimate never exceeds the real remaining time by a stated margin.
  - Define where the guard persists the carried state, with the project hash it came from.
  - Cap the halt threshold with an absolute ceiling the guard compiles in, about 3,600 seconds.
  - Track battery replacement and a six-monthly look at trial 1 outside the model, as a Home Assistant reminder.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from workbench import setup_workspace

ws = setup_workspace("rackmains/ups/battery", {"kind": "embedded", "language": "rust"})
CONSTANTS = {
    "guard_shed_max_seconds": 120,
    "guard_halt_max_seconds": 450,
    "guard_mains_stable_seconds": 300,
    "guard_halt_delay_seconds": 120,
}

## Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-data))

In [ ]:
def golden_rows():
    clock = "clock_epoch_seconds"
    util = "output_utilisation_pct"
    battery = "ups_on_battery"
    replace = "battery_replace"
    state = "model_state_in"
    return {
        "case": ["first call with no state", "mid-drain with state", "replace flag raised", "required input missing"],
        "expect": ["ok", "ok", "ok", "error"],
        clock: [1793355300.0, 1793355330.0, 1793355360.0, 1793355390.0],
        util: [13, 13, 12, None],
        battery: [1, 1, 0, 1],
        replace: [0, 0, 1, 0],
        state: [None, "0.0 1235.0 0.0 1.0 1.0 1793355329.0", "0.0 0.0 600.0 0.0 1.0 1793355359.0", None],
    }


golden = golden_rows()

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench import package_workspace

package_workspace(ws, "static", golden, trained_on="none, the rollout A starting values", notes="a conservative starting set, not fitted", constants=CONSTANTS, sequence="baseline calls")

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
from workbench import register_workspace

register_workspace(ws)